# 🤗 Template 3 : transfer learning — DistilBERT

*Préparé par Claude le 19/10/2026 pour le projet `01-AT&T_spam_detector.ipynb`. Aucun code de solution ici : c'est toi qui écris.*

## Les 4 templates du projet

| # | Phase | Notebook | Entrée | Sortie |
|---|---|---|---|---|
| 1 | Préparation des données | `02-Template_1_preparation_donnees` | `src/spam.csv` | `src/train.csv`, `val.csv`, `test.csv` |
| 2 | Modèle simple : embeddings + petit réseau | `03-Template_2_modele_embeddings` | les 3 CSV | `scores/scores_embeddings.json`, `scores/predictions_embeddings.csv` |
| **3** | **Transfer learning : DistilBERT** (ce notebook) | `04-Template_3_transfer_learning` | les 3 CSV | `scores/scores_transfer.json`, `scores/predictions_transfer.csv` |
| 4 | Évaluation et conclusion | `05-Template_4_evaluation_conclusion` | les fichiers de `scores/` | le tableau final du livrable |

Ce notebook répond au second conseil de l'énoncé : **« Transfer learning »**. 3 617 SMS, c'est peu pour apprendre l'anglais de zéro. On part donc d'un modèle qui le connaît déjà.

## Le principe

```
                 ┌──────────── déjà appris par Google/Hugging Face, sur Wikipédia + des milliers de livres ────────────┐
"FREE entry NOW!" → tokenizer DistilBERT → [101, 2489, 4443, 2085, 999, 102] → DistilBERT (66 millions de paramètres) → vecteur du SMS
                                                                                                                        │
                                                                         tête de classification (NOUVELLE, au hasard) ◄──┘
                                                                                                                        │
                                                                                                                  P(ham), P(spam)
```

**Fine-tuning** : on reprend tous les poids de DistilBERT, on ajoute une petite tête de classification, et on réentraîne **le tout, doucement** (petit *learning rate*, 2 à 3 époques). Le modèle n'apprend pas l'anglais : il apprend seulement à **reconnaître un spam**, à partir de ce qu'il sait déjà de la langue.

| | Template 2 (embeddings) | Template 3 (DistilBERT) |
|---|---|---|
| connaissance de départ | aucune | la langue anglaise |
| découpage des mots | le nôtre (`\w+`) | celui du modèle (sous-mots) |
| ordre des mots | ignoré (moyenne) | pris en compte (*attention*) |
| taille | ~100 000 paramètres | ~66 millions |

## Comment s'en servir

Même rythme qu'au template 1 :

1. **🎯 Objectif** : ce qu'on veut obtenir, et pourquoi ;
2. **🧰 Outils** : les fonctions utiles, leur rôle, leurs arguments importants (parfois avec une petite démo dans un autre contexte) ;
3. **✍️ À toi** : une cellule avec le plan en commentaires, que tu remplis ;
4. **✅ Contrôle** : une cellule déjà écrite qui vérifie ton travail. ✅ = c'est bon, `AssertionError` = relis l'étape.

Les noms de variables imposés sont ceux qu'attendent les contrôles. Lance toujours les cellules **dans l'ordre**.

⏱️ **Prévois du temps** : le premier lancement télécharge le modèle (~270 Mo), et l'entraînement prend **10 à 20 minutes** sur un processeur (une à deux sur une carte graphique).

---
# Étape 0 — Les imports
Rien à écrire : lance la cellule.

In [ ]:
import html
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding,
                          Trainer, TrainingArguments, set_seed)
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)

DOSSIER_SRC = Path("src")
DOSSIER_SCORES = Path("scores")    # partagé sur GitHub pour le template 4
DOSSIER_MODELES = Path("models")   # ignoré par Git (le modèle pèse ~270 Mo)
NOM_MODELE = "distilbert/distilbert-base-uncased"   # « uncased » : le modèle ne distingue pas majuscules et minuscules
SEED = 42

set_seed(SEED)   # fixe d'un coup les graines de Python, numpy et PyTorch
print("prêt — carte graphique disponible :", torch.cuda.is_available())

---
# Étape 1 — Charger et décoder le HTML

🎯 Lire les trois CSV dans `train_df`, `val_df`, `test_df`, puis **décoder les codes HTML** de la colonne `text` : `&lt;#&gt;` devient `<#>`, `&amp;` devient `&`.

❓ Pourquoi ici, alors qu'au template 2 on avait remplacé `&lt;#&gt;` par `numtoken` ? DistilBERT a appris l'anglais sur du texte **normal**, pas sur du code HTML : `<#>` lui parle plus que `&lt;#&gt;`. Avec un modèle pré-entraîné, on donne un texte aussi naturel que possible.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `html.unescape(texte)` | décode les codes HTML | — |
| `df["text"].apply(fonction)` | applique une fonction à chaque texte | — |

In [ ]:
print(html.unescape("Tom &amp; Jerry &lt;3"))   # démo

In [ ]:
# ✍️ À toi
# 1. train_df, val_df, test_df = les 3 CSV
# 2. dans chacun, remplace la colonne text par sa version décodée (html.unescape)
# 3. affiche un SMS qui contenait "&lt;#&gt;", avant et après

In [ ]:
# ✅ Contrôle
assert (len(train_df), len(val_df), len(test_df)) == (3617, 776, 776)
for nom, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    assert not df["text"].str.contains("&lt;", regex=False).any(), f"{nom} : il reste des codes HTML"
print("✅ étape 1")

---
# Étape 2 — Découvrir le tokenizer

🎯 Chaque modèle pré-entraîné vient avec **son** tokenizer : il faut découper les textes **exactement** comme pendant son pré-entraînement, sinon les numéros ne correspondent plus aux bons mots.

Ce que tu vas observer :
- `[CLS]` au début et `[SEP]` à la fin : des tokens spéciaux ajoutés automatiquement. Le vecteur de `[CLS]` sert à résumer la phrase pour la classification ;
- des **sous-mots** : un mot inconnu du vocabulaire est découpé en morceaux connus (`txt` → `tx` + `##t`). Résultat : **aucun mot n'est jamais inconnu**, contrairement au `<unk>` du template 2 ;
- `attention_mask` : 1 pour un vrai token, 0 pour le remplissage (comme notre masque du template 2).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `AutoTokenizer.from_pretrained(NOM_MODELE)` | télécharge le tokenizer du modèle | — |
| `tokenizer(texte)` | texte → `input_ids` + `attention_mask` | — |
| `tokenizer.convert_ids_to_tokens(ids)` | numéros → tokens lisibles | — |
| `tokenizer.vocab_size` | taille du vocabulaire | — |

In [ ]:
# ✍️ À toi
# 1. tokenizer = le tokenizer de NOM_MODELE
# 2. encode un spam de train_df : affiche input_ids, attention_mask et les tokens lisibles
# 3. affiche la taille du vocabulaire du tokenizer

In [ ]:
# ✅ Contrôle
ids = tokenizer("hello world")["input_ids"]
assert ids[0] == tokenizer.cls_token_id and ids[-1] == tokenizer.sep_token_id
assert tokenizer.vocab_size == 30522
print("✅ étape 2")

👀 **Observe** : comment sont découpés les numéros de téléphone, `txt`, `£`, `<#>` ? Le vocabulaire de DistilBERT (30 522) est-il plus grand ou plus petit que celui du template 2 ? Pourquoi n'a-t-il pourtant jamais de mot inconnu ?

---
# Étape 3 — Choisir la longueur maximale (en tokens)

🎯 Même raisonnement qu'au template 1 (étape 3.c), mais en **tokens** : les sous-mots et la ponctuation font qu'un SMS a plus de tokens que de mots.

Choisir `MAX_LEN_TOKENS` qui couvre presque tous les SMS. Ici l'enjeu est encore plus fort : le temps de calcul de DistilBERT grandit **plus vite** que la longueur (chaque token « regarde » tous les autres).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `len(tokenizer(texte)["input_ids"])` | nombre de tokens d'un SMS (avec `[CLS]` et `[SEP]`) | — |
| `.quantile(0.95)`, `.max()` | comme au template 1 | — |

In [ ]:
# ✍️ À toi
# 1. n_tokens = une Series : le nombre de tokens de chaque SMS de train_df
# 2. affiche la médiane, le 95e centile et le maximum
# 3. MAX_LEN_TOKENS = ton choix (un nombre rond un peu au-dessus du 95e centile)
# 4. affiche la part des SMS de train qui tiennent entièrement dans MAX_LEN_TOKENS

In [ ]:
# ✅ Contrôle
assert n_tokens.quantile(0.95) == 55, f"95e centile attendu : 55, obtenu {n_tokens.quantile(0.95)}"
assert n_tokens.max() == 221
assert n_tokens.quantile(0.95) <= MAX_LEN_TOKENS <= 128, "MAX_LEN_TOKENS doit couvrir le 95e centile sans exagérer"
print("✅ étape 3")

✍️ **À écrire** : quelle valeur choisis-tu ? Pourquoi un SMS a-t-il plus de tokens que de mots ?

*Ta réponse :*

---
# Étape 4 — Préparer les datasets Hugging Face

🎯 L'outil d'entraînement de Hugging Face (`Trainer`) attend des `Dataset` Hugging Face, pas des DataFrames. Il faut :

1. convertir chaque DataFrame (`text`, `label`) en `Dataset` ;
2. tokeniser tous les textes, **coupés** à `MAX_LEN_TOKENS`.

On ne **complète pas** (pas de *padding*) ici : un « collator » le fera à l'entraînement, lot par lot, juste à la longueur du plus long SMS **du lot**. Un lot de SMS courts va ainsi beaucoup plus vite.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `Dataset.from_pandas(df)` | DataFrame → Dataset | `preserve_index=False` |
| `ds.map(fonction, batched=True)` | applique la fonction à des paquets de lignes ; ses sorties deviennent de nouvelles colonnes | `batched=True` : la fonction reçoit **une liste** de textes |
| `tokenizer(liste_de_textes, ...)` | tokenise une liste d'un coup | `truncation=True`, `max_length=MAX_LEN_TOKENS` |

In [ ]:
# ✍️ À toi
# 1. def tokeniser_lot(lot) : renvoie le tokenizer appliqué à lot["text"] (coupé à MAX_LEN_TOKENS)
# 2. train_ds, val_ds, test_ds = les 3 DataFrames convertis puis tokenisés avec .map
# 3. affiche train_ds et le premier exemple

In [ ]:
# ✅ Contrôle
for nom, ds, n in [("train", train_ds, 3617), ("val", val_ds, 776), ("test", test_ds, 776)]:
    assert len(ds) == n, f"{nom} : {len(ds)} lignes"
    assert {"input_ids", "attention_mask", "label"} <= set(ds.column_names), f"{nom} : colonnes {ds.column_names}"
assert max(len(ids) for ids in train_ds["input_ids"]) == MAX_LEN_TOKENS, "les SMS longs doivent être coupés à MAX_LEN_TOKENS"
assert min(len(ids) for ids in train_ds["input_ids"]) < MAX_LEN_TOKENS, "pas de padding ici : le collator s'en charge"
print("✅ étape 4")

---
# Étape 5 — Charger le modèle pré-entraîné

🎯 Charger DistilBERT **avec une tête de classification à 2 sorties** (ham, spam).

Un rapport de chargement (*LOAD REPORT*) va s'afficher, avec deux sortes de lignes. **C'est normal** :
- `UNEXPECTED` (`vocab_transform`, `vocab_projector`...) : des morceaux du modèle d'origine qui servaient à deviner des mots masqués pendant son pré-entraînement. On n'en a pas besoin : ils sont laissés de côté ;
- `MISSING` (`pre_classifier`, `classifier`) : la **tête de classification**, toute neuve, initialisée au hasard. Elle n'a encore rien appris : c'est exactement ce que l'étape 7 va faire.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `AutoModelForSequenceClassification.from_pretrained(...)` | DistilBERT + une tête de classification | `num_labels=2`, `id2label={0: "ham", 1: "spam"}`, `label2id={"ham": 0, "spam": 1}` |
| `sum(p.numel() for p in modele.parameters())` | nombre de paramètres | — |

In [ ]:
# ✍️ À toi
# 1. modele = DistilBERT avec une tête à 2 classes (et les noms ham / spam)
# 2. n_parametres = son nombre de paramètres ; affiche-le

In [ ]:
# ✅ Contrôle
assert modele.config.num_labels == 2 and modele.config.id2label[1] == "spam"
assert n_parametres > 60_000_000
print(f"✅ étape 5 — {n_parametres / 1e6:.0f} millions de paramètres")

👀 **Observe** : combien de fois plus de paramètres qu'au template 2 ? Garde ce chiffre pour la comparaison du template 4.

---
# Étape 6 — La fonction de métriques

🎯 Pendant l'entraînement, le `Trainer` évaluera le modèle sur val à chaque époque. Il faut lui dire **quoi** calculer : écrire `compute_metrics(eval_pred)`.

Le `Trainer` lui donne `eval_pred`, qui contient deux tableaux :
- `logits` : forme `(nombre de SMS, 2)`, un score pour ham et un pour spam ;
- `labels` : les vrais labels.

La classe prédite est la colonne au score le plus haut. La fonction renvoie un **dictionnaire** `{"accuracy": ..., "precision": ..., "recall": ..., "f1": ...}` (classe spam).

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `logits, labels = eval_pred` | sépare les deux tableaux | — |
| `np.argmax(logits, axis=1)` | numéro de la colonne max, pour chaque ligne | `axis=1` |
| `precision_score(..., zero_division=0)` | évite un avertissement si aucun spam n'est prédit | `zero_division=0` |

In [ ]:
# ✍️ À toi
# def compute_metrics(eval_pred) : renvoie le dictionnaire des 4 métriques

In [ ]:
# ✅ Contrôle : 3 SMS factices — prédits ham, spam, spam ; vrais ham, spam, ham
m = compute_metrics((np.array([[2., 0.], [0., 2.], [0., 2.]]), np.array([0, 1, 0])))
assert set(m) == {"accuracy", "precision", "recall", "f1"}
assert round(m["precision"], 3) == 0.5 and m["recall"] == 1 and round(m["f1"], 3) == 0.667, m
print("✅ étape 6")

---
# Étape 7 — Fine-tuner

🎯 Le `Trainer` fait toute la boucle du template 2 à ta place (lots, `zero_grad`, `backward`, `step`, évaluation). Il suffit de le configurer.

**a. `TrainingArguments`** : les réglages.

| Argument | Valeur | Pourquoi |
|---|---|---|
| `output_dir` | `str(DOSSIER_MODELES / "distilbert_entrainement")` | dossier de travail du Trainer |
| `num_train_epochs` | `3` | le modèle sait déjà l'anglais : peu d'époques suffisent |
| `learning_rate` | `2e-5` | **très petit** (50 fois moins qu'au template 2) : on ajuste doucement, sans effacer ce que le modèle sait |
| `per_device_train_batch_size` | `16` | lots plus petits : le modèle est lourd en mémoire |
| `per_device_eval_batch_size` | `64` | en évaluation, pas de gradients : on peut faire plus gros |
| `eval_strategy` | `"epoch"` | évalue sur val à la fin de chaque époque |
| `logging_strategy` | `"epoch"` | note la loss de train à chaque époque |
| `save_strategy` | `"no"` | pas de sauvegarde intermédiaire (on enregistre à la fin, étape 11) |
| `report_to` | `"none"` | pas d'envoi vers un service de suivi en ligne |
| `seed` | `SEED` | reproductible |

**b. `Trainer`** : rassemble `model`, `args`, `train_dataset`, `eval_dataset` (val !), `processing_class=tokenizer`, `data_collator=DataCollatorWithPadding(tokenizer)` (le remplissage lot par lot), `compute_metrics`.

**c. `trainer.train()`**, en mesurant la durée dans `temps_entrainement` (comme au template 2).

⏱️ C'est la cellule longue : 10 à 20 minutes sur processeur. Un tableau de suivi s'affiche, avec une ligne par époque.

Quelques avertissements peuvent apparaître dans ce notebook, **sans gravité** : `pin_memory` (pas de carte graphique), `symlinks` (une limite de Windows pour le cache des modèles), `HF_TOKEN` (téléchargement sans compte Hugging Face), `IProgress not found` (barre de progression simplifiée).

In [ ]:
# ✍️ À toi
# 1. args = TrainingArguments(...) avec les réglages du tableau
# 2. trainer = Trainer(...)
# 3. debut = time.perf_counter() ; trainer.train() ; temps_entrainement = durée en secondes

In [ ]:
# ✅ Contrôle
evals = [h for h in trainer.state.log_history if "eval_f1" in h]
assert len(evals) == args.num_train_epochs, "une évaluation par époque attendue : vérifie eval_strategy"
assert evals[-1]["eval_f1"] > 0.90, f"F1 val trop bas : {evals[-1]['eval_f1']:.3f} — vérifie le learning_rate"
print(f"✅ étape 7 — F1 val à la dernière époque : {evals[-1]['eval_f1']:.3f}")

---
# Étape 8 — Lire les courbes d'apprentissage

🎯 Le `Trainer` garde tout dans `trainer.state.log_history` : une liste de dictionnaires, avec à chaque époque une entrée `loss` (train) et une entrée `eval_loss`, `eval_f1`... (val). Les mettre dans un tableau et tracer, comme au template 2.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `[h for h in liste if "cle" in h]` | garde les dictionnaires qui ont une clé | — |
| `pd.DataFrame(liste)` | liste de dictionnaires → tableau | — |
| `pd.merge(a, b, on="epoch")` | recolle deux tableaux sur l'époque | `on="epoch"` |

In [ ]:
# ✍️ À toi
# 1. courbes = un DataFrame avec les colonnes epoch, loss (train), eval_loss, eval_f1
# 2. trace loss et eval_loss en fonction de epoch

✍️ **À écrire** : y a-t-il sur-apprentissage ? Compare à la vitesse d'apprentissage du template 2 : en combien d'époques chaque modèle atteint-il un bon F1 ?

*Ta réponse :*

---
# Étape 9 — Évaluer sur validation et choisir le seuil

🎯 Comme au template 2, mais il faut d'abord transformer les 2 logits en **probabilité de spam** :

```
logits (ham, spam) = (-1.2, 2.3)  ──softmax──►  (0.03, 0.97)  ──colonne 1──►  P(spam) = 0.97
```

Puis : rapport au seuil 0,5, tableau des seuils de 0,10 à 0,90, choix de `SEUIL` **sur val**.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `trainer.predict(ds).predictions` | les logits de tous les SMS du dataset | — |
| `torch.softmax(torch.tensor(logits), dim=1)` | logits → probabilités (chaque ligne somme à 1) | `dim=1` |
| `[:, 1].numpy()` | colonne spam, en numpy | — |
| le reste | comme au template 2 (étape 9) | — |

In [ ]:
# ✍️ À toi
# 1. def predire_probas(ds) : predict → softmax → colonne spam (tableau numpy)
# 2. probas_val = predire_probas(val_ds) ; au seuil 0,5 : classification_report et confusion_matrix
# 3. tableau_seuils : colonnes "seuil", "precision", "recall", "f1" pour les seuils de 0,10 à 0,90 ; trace-le
# 4. SEUIL = le meilleur seuil sur val (ou ton choix argumenté)

In [ ]:
# ✅ Contrôle
assert probas_val.shape == (776,) and 0 <= probas_val.min() and probas_val.max() <= 1
assert {"seuil", "precision", "recall", "f1"} <= set(tableau_seuils.columns)
assert 0 < SEUIL < 1
assert f1_score(val_df["label"], probas_val >= SEUIL) > 0.90
print(f"✅ étape 9 — F1 val au seuil {SEUIL} : {f1_score(val_df['label'], probas_val >= SEUIL):.3f}")

✍️ **À écrire** : quel seuil, et pourquoi ? La courbe des seuils est-elle plus « plate » qu'au template 2 ? Qu'est-ce que ça dit de la confiance du modèle ?

*Ta réponse :*

---
# Étape 10 — Évaluation finale sur test (une seule fois)

🎯 Même chose qu'au template 2 : un dictionnaire `scores_transfer` avec exactement les clés
`"modele"`, `"seuil"`, `"accuracy"`, `"precision"`, `"recall"`, `"f1"`, `"n_parametres"`, `"temps_entrainement_s"`.

In [ ]:
# ✍️ À toi
# 1. probas_test = predire_probas(test_ds) ; preds_test au SEUIL
# 2. scores_transfer = {"modele": "DistilBERT (transfer learning)", ... les 8 clés}
# 3. affiche scores_transfer et la matrice de confusion de test

In [ ]:
# ✅ Contrôle
cles = {"modele", "seuil", "accuracy", "precision", "recall", "f1", "n_parametres", "temps_entrainement_s"}
assert set(scores_transfer) == cles, f"clés attendues : {cles}"
assert all(0 <= scores_transfer[k] <= 1 for k in ["accuracy", "precision", "recall", "f1"])
assert abs(scores_transfer["f1"] - f1_score(test_df["label"], probas_test >= SEUIL)) < 1e-9, "le F1 doit être calculé au SEUIL choisi"
assert scores_transfer["f1"] > 0.85
print("✅ étape 10")

---
# Étape 11 — Enregistrer pour le template 4

🎯 Comme au template 2 :

| Fichier | Contenu | Sur GitHub ? |
|---|---|---|
| `scores/scores_transfer.json` | `scores_transfer` | ✅ |
| `scores/predictions_transfer.csv` | `text`, `label`, `proba` pour chaque SMS de test | ✅ |
| `models/distilbert_spam/` | le modèle **et** son tokenizer | ❌ (~270 Mo) |

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `trainer.save_model(chemin)` | enregistre le modèle, et le tokenizer donné en `processing_class` | `str(...)` |
| le reste | comme au template 2 (étape 11) | — |

In [ ]:
# ✍️ À toi
# 1. crée DOSSIER_SCORES ; écris scores_transfer dans DOSSIER_SCORES / "scores_transfer.json"
# 2. écris test_df + une colonne proba dans DOSSIER_SCORES / "predictions_transfer.csv" (sans index)
# 3. enregistre le modèle dans DOSSIER_MODELES / "distilbert_spam"

In [ ]:
# ✅ Contrôle
with open(DOSSIER_SCORES / "scores_transfer.json", encoding="utf-8") as f:
    assert json.load(f) == scores_transfer
preds = pd.read_csv(DOSSIER_SCORES / "predictions_transfer.csv")
assert list(preds.columns) == ["text", "label", "proba"] and len(preds) == 776
assert (preds["label"] == test_df["label"]).all(), "les lignes doivent rester dans l'ordre de test.csv"
assert (DOSSIER_MODELES / "distilbert_spam" / "config.json").exists()
print("✅ étape 11 : résultats prêts pour le template 4")

---
# Étape 12 — Essayer sur de nouveaux SMS

🎯 `predire_sms(texte)` → probabilité de spam d'un SMS inventé. Mêmes SMS qu'au template 2, pour comparer.

🧰 Outils

| Outil | Rôle | Arguments importants |
|---|---|---|
| `tokenizer(texte, return_tensors="pt")` | tokenise directement en tenseurs PyTorch | `return_tensors="pt"` |
| `.to(modele.device)` | place les tenseurs au même endroit que le modèle (processeur ou carte graphique) | — |
| `modele(**entrees).logits` | prédiction ; `**` déballe le dictionnaire `input_ids` + `attention_mask` | — |

In [ ]:
# ✍️ À toi
# 1. def predire_sms(texte) : eval, no_grad, tokenizer, modele, softmax → P(spam) en nombre Python
# 2. affiche la probabilité et la décision au SEUIL pour chaque SMS ; ajoute les tiens
sms_inventes = [
    "Hey, are we still meeting for lunch tomorrow?",
    "URGENT! You have won a 1000 cash prize. Call 09061234567 now to claim",
    "Ok see u later lol",
    "FREE entry: txt WIN to 80086 to receive your reward",
]

In [ ]:
# ✅ Contrôle
assert 0 <= predire_sms("hello") <= 1
assert predire_sms(sms_inventes[3]) > predire_sms(sms_inventes[0])
print("✅ étape 12 — template 3 terminé")

---
# 📝 Pour le livrable

| Décision | Ta justification |
|---|---|
| pourquoi le transfer learning (conseil de l'énoncé) | |
| modèle choisi : DistilBERT *uncased* | |
| décodage HTML au lieu de `numtoken` | |
| `MAX_LEN_TOKENS` | |
| `learning_rate = 2e-5`, 3 époques | |
| seuil retenu | |
| **scores sur test** (précision, recall, F1 spam) | |
| nombre de paramètres et durée d'entraînement | |